In [182]:
using LowLevelFEM, LinearAlgebra

openGeometry("I-szelveny-2.geo")

In [183]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u)

Problem("I-szelveny-2", :VectorField, 3, 3, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 112833, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false)

In [184]:
cs = CoordinateSystem([-1, 0, 0], [0, -1, 0])
Q = rotateNodes(U, "left", cs)
cs1 = BoundaryCondition("left", ux=0, uy=0, uz=0)
dofs = constrainedDoFs(U, [cs1])
for i in dofs
    Q.T[i, i] = -1
end

In [185]:
K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U), Ω="body")
K = Q' * K * Q

sparse([1, 2, 3, 115, 116, 117, 754, 755, 756, 757  …  328512, 331114, 331115, 331116, 333739, 333740, 333741, 338497, 338498, 338499], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  338499, 338499, 338499, 338499, 338499, 338499, 338499, 338499, 338499, 338499], [247285.1253861655, 95148.24721188965, 85813.15567468441, -63211.41046427257, 25601.128729018652, 26441.31459153177, 15188.058498142398, 9915.464321528005, 21294.786939598223, -16218.70177549768  …  -65887.44308570186, 37774.793309963796, -176999.1084781225, 114329.64593834885, 50699.523182577104, 46333.919351545774, 104317.7708023529, -135342.4472554631, 278146.97906030354, 3.3278486162643894e6], 338499, 338499)

In [186]:
fx = ScalarField(U, "right", (x,y,z)->-y/1000)
fy = ScalarField(U, "right", (x,y,z)->(x-100)/1000)

elementwise ScalarField
[[-0.15; -0.14655172413793105; -0.14754798168869424;;], [-0.15; -0.15; -0.14754798168869424;;], [-0.05344827586206898; -0.05; -0.05245201831130587;;], [-0.05; -0.05; -0.05245201831130587;;], [-0.15; -0.15; -0.14754654142298052;;], [-0.14653846153846153; -0.15; -0.14754654142298052;;], [-0.105; -0.10846153846153847; -0.10715658661381491;;], [-0.105; -0.105; -0.10217137716980063;;], [-0.10217137716980063; -0.105; -0.10198436333744064;;], [-0.10198436333744064; -0.105; -0.10485728783182978;;]  …  [-0.06726640803258323; -0.06896551724137931; -0.07068965517241378;;], [-0.11895166023697021; -0.11724137931034487; -0.11550183037841717;;], [-0.06896551724137939; -0.06726698504587185; -0.07068965517241389;;], [-0.12585478237059408; -0.12413793103448277; -0.12240170679307408;;], [-0.07414521762940592; -0.07586206896551723; -0.07759829320692589;;], [-0.1396455586079843; -0.13793103448275865; -0.1361878592745384;;], [-0.060354441392015686; -0.062068965517241365; -0.063812140

In [187]:
f = ∫(U ⋅ [fx, fy, 0], Γ="right")

nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [188]:
showDoFResults(f, name="f");

In [189]:
bc = BoundaryCondition("right", ux=fx, uy=fy, uz=0)
bc2 = BoundaryCondition("left", uz=0)

BoundaryCondition("left", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uz => 0))

In [190]:
per = MPC(master="right", slave="left");

In [191]:
u = solveField(K, 0f, support=[bc, bc2], mpc=[per]);
u = Q * u;

In [192]:
showDoFResults(u, :uvec, name="displacement", factor=1);

In [193]:
u1 = nodesToElements(u, onPhysicalGroup="mid")
ε = (u1 ∘ ∇ + ∇ ∘ u1) / 2

E = mat.E
ν = mat.μ
I = unitTensor(ε)

σ = E / (1+ν) * (ε + ν / (1 - 2ν) * trace(ε) * I);

In [194]:
R = nodePositionVector(U)
R = nodesToElements(R, onPhysicalGroup="mid")

n2 = normalVector(U, "mid")

σ2 = elementsToElements(σ, onPhysicalGroup="mid")

τ2 = σ2 * n2;

In [195]:
showElementResults(τ2, name="shear stress");

In [196]:
showDoFResults(n2, name="normal vector");

In [197]:
openPostProcessor()